# Regime-Aware Machine Learning for Event-Driven Trading:
# A Walk-Forward Analysis on Equity Markets

---

## Progress Report — Pattern Detection Phase

**Author:** Zeineb Turki  
**Date:** March 2026  
**Phase:** 3 of 9 — Pattern Detection Tuning & Validation (Complete)

---

### Objective

This project investigates whether combining **Hidden Markov Model (HMM) market regime detection** with **event-driven technical signal selection** improves out-of-sample risk-adjusted trading performance compared to regime-agnostic and non-event-based baselines.

Unlike dense prediction models that evaluate every trading day, this system is **event-driven**: machine learning models are only invoked around meaningful technical events such as support/resistance touches, triangle breakouts, channel boundary approaches, and multiple tops/bottoms.

### Scope of this notebook

This notebook documents the completed **pattern detection phase** of the project:

1. The SPY OHLCV data pipeline
2. The ATR-based event detection redesign
3. The tuning and validation of all four pattern detectors
4. Before/after comparisons showing improved signal quality
5. Why the detection layer is now ready for HMM regime integration

This notebook does **not** cover HMM fitting, feature engineering, ML modelling, or backtesting — those are subsequent phases.

In [ ]:
import sys, os

# ── Colab setup ──────────────────────────────────────────────────────────
# On Google Colab the repository must be cloned first.
# Locally (Jupyter / VS Code) it is already on disk.
if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')

    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')

    os.system(f'{sys.executable} -m pip install -q hmmlearn scikit-learn seaborn statsmodels')

else:
    # Local: walk upward to find the directory that contains src/patterns/
    PROJ_ROOT = None
    _cand = os.path.abspath('.')
    for _ in range(6):
        if os.path.isdir(os.path.join(_cand, 'src', 'patterns')):
            PROJ_ROOT = _cand
            break
        # Also peek one level deeper (handles the outer repo root)
        for _sub in os.listdir(_cand):
            _nested = os.path.join(_cand, _sub)
            if os.path.isdir(_nested) and os.path.isdir(os.path.join(_nested, 'src', 'patterns')):
                PROJ_ROOT = _nested
                break
        if PROJ_ROOT:
            break
        _cand = os.path.dirname(_cand)

    if PROJ_ROOT is None:
        raise RuntimeError('Could not locate src/patterns/. Open from inside the project repo.')

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)
print(f'Project root: {PROJ_ROOT}')

# ── Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.load_data import load_spy
from src.data.utils import compute_atr
from src.patterns.scanner import scan_all_patterns
from src.patterns.validation import EventValidator

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'figure.figsize': (14, 4.5),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})

print('All imports successful.')

---

## 1. Data Overview

The project uses daily OHLCV data for the **SPDR S&P 500 ETF Trust (SPY)**, downloaded from Yahoo Finance via `yfinance`. SPY is chosen because it is the most liquid equity ETF, has tight spreads, and provides a long, clean history suitable for walk-forward analysis.

In [ ]:
df = load_spy()

summary = pd.DataFrame({
    'Metric': ['Ticker', 'Start date', 'End date', 'Trading days', 'Columns',
               'Open range', 'Close range', 'Volume (mean)'],
    'Value': ['SPY', str(df.index[0].date()), str(df.index[-1].date()),
              f'{len(df):,}', ', '.join(df.columns),
              f'${df["Open"].min():.2f} – ${df["Open"].max():.2f}',
              f'${df["Close"].min():.2f} – ${df["Close"].max():.2f}',
              f'{df["Volume"].mean():,.0f}']
})
display(summary.style.hide(axis='index'))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# --- Panel 1: Close price ---
axes[0].plot(df.index, df['Close'], color='#1f77b4', linewidth=0.8)
axes[0].set_ylabel('Price ($)')
axes[0].set_title('SPY Daily Close Price')

# --- Panel 2: Daily returns ---
returns = df['Close'].pct_change() * 100
axes[1].bar(df.index, returns, width=1, color=np.where(returns >= 0, '#2ca02c', '#d62728'), alpha=0.6)
axes[1].set_ylabel('Daily Return (%)')
axes[1].set_title('Daily Returns')
axes[1].set_ylim(-12, 12)

# --- Panel 3: ATR(14) ---
atr = compute_atr(df, window=14)
axes[2].plot(df.index, atr, color='#ff7f0e', linewidth=0.8)
axes[2].set_ylabel('ATR ($)')
axes[2].set_title('Average True Range (14-day)')
axes[2].set_xlabel('Date')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Data observations

- SPY spans **16 years** of trading, covering multiple market regimes: the post-2008 recovery, the 2020 COVID crash, and the 2022 bear market.
- ATR shows clear regime shifts in volatility, spiking during crises (2011, 2018, 2020, 2022) and compressing during calm periods.
- This volatility variation is the reason we use **ATR-based adaptive thresholds** in all pattern detectors — a fixed-dollar threshold would be meaningless across a price range of $80 to $600+.

---

## 2. Why Dense Event Detection Was a Problem

### The original issue

In the initial implementation, the four pattern detectors (support/resistance, triangles, channels, multiple tops/bottoms) were applied with default parameters. The result was an **event rate of approximately 26.5%** — meaning more than one in four trading days was flagged as a pattern event.

This is fundamentally incompatible with the project's **event-driven** thesis:

| Requirement | Dense detection (26.5%) | Event-driven goal |
|-------------|------------------------|-------------------|
| Signal selectivity | Low — fires on noise | High — fires on structure |
| Visual meaningfulness | Many false positives | Matches chart intuition |
| Clustering | Same event fires 5–10 consecutive days | One signal per event |
| ML label quality | Contaminated by noise | Clean training signal |
| Backtest realism | Overtrading | Realistic trade frequency |

### The three core problems

1. **No cooldown**: once a condition was met, it fired every day until the condition expired. A single support touch would generate 5–10 consecutive signals.

2. **No structural validation**: detectors used simple rolling-window statistics without checking whether the structure was *visually meaningful* — e.g., a "channel" was detected from any trending data, regardless of whether price actually bounced between parallel trendlines.

3. **Resistance in uptrends**: the support/resistance detector flagged nearly every day in a bull market, because the rolling 50-bar high ("resistance") was continuously rising with price.

### Supervisor feedback

The project supervisor specifically noted:
- Detected signals should match what a **human trader would reasonably identify** on a chart
- The parametrization needed further tuning
- Reference cases (descending triangle upper-limit tests, well-defined trend channels) should guide calibration

### The role of ATR

The **Average True Range** (ATR) is the shared volatility measure used across all detectors. Every proximity band, breakout threshold, and channel-width constraint is expressed as a multiple of ATR(14), so that patterns automatically adapt to the current volatility regime. This design principle was established in Phase 2 and retained throughout the tuning process.

In [ ]:
# Demonstrate the original (pre-tuning) event density
# We reconstruct the old behavior by running detectors WITHOUT the new filters

from src.patterns.support_resistance import calculate_support_resistance

# Old S/R: no stability filter, no cooldown
df_old_sr = df.copy()
df_old_sr['resistance'] = df_old_sr['High'].rolling(50).max()
df_old_sr['support'] = df_old_sr['Low'].rolling(50).min()
atr_vals = compute_atr(df_old_sr)
band = 0.3 * atr_vals
old_nr = (df_old_sr['Close'] - df_old_sr['resistance']).abs() <= band
old_ns = (df_old_sr['Close'] - df_old_sr['support']).abs() <= band
old_sr_count = (old_nr | old_ns).sum()

# Show old vs new counts in a comparison table
old_counts = {
    'Support/Resistance': 672,
    'Channels': 235,
    'Triangles': 80,
    'Multi Top/Bottom': 87,
    'TOTAL': 1067,
}

old_rates = {k: f'{v/len(df)*100:.1f}%' for k, v in old_counts.items()}

comparison = pd.DataFrame({
    'Detector': old_counts.keys(),
    'Events (before)': old_counts.values(),
    'Rate (before)': old_rates.values(),
})

print('Event detection BEFORE tuning (Iteration 2):')
display(comparison.style.hide(axis='index'))

In [ ]:
# Demonstrate clustering in old near_resistance
old_nr_dates = df.index[old_nr]
old_nr_diffs = old_nr_dates.to_series().diff().dt.days

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(old_nr_diffs.dropna(), bins=range(0, 52), color='#d62728', alpha=0.7, edgecolor='white')
ax.set_xlabel('Days between consecutive near_resistance signals')
ax.set_ylabel('Count')
ax.set_title('Clustering problem: near_resistance inter-signal gaps (before tuning)')
ax.axvline(1, color='black', linestyle='--', alpha=0.5, label='Consecutive day (gap=1)')
ax.legend()
plt.tight_layout()
plt.show()

pct_consec = (old_nr_diffs == 1).sum() / len(old_nr_diffs.dropna()) * 100
pct_within5 = (old_nr_diffs <= 5).sum() / len(old_nr_diffs.dropna()) * 100
print(f'Consecutive-day signals: {pct_consec:.0f}% of all near_resistance events')
print(f'Within 5 days:           {pct_within5:.0f}% of all near_resistance events')
print()
print('This means the same resistance level was firing day after day.')
print('A human trader would see ONE event; the detector saw 5-10.')

### Section conclusion

The original detection layer was producing far too many signals, most of which were consecutive-day repetitions of the same underlying pattern. This would contaminate any downstream ML model with redundant, correlated training examples. Fixing this was a prerequisite for all subsequent phases.

---

## 3. Detector Redesign — Detailed Changes

Each of the four detectors was tuned independently. The guiding principles were:

1. **One signal per event** — cooldown periods prevent consecutive-day firing
2. **Structural validation** — detectors must confirm that the pattern has real structure, not just statistical coincidence
3. **ATR-adaptive thresholds** — all distance measures scale with current volatility
4. **Actionable timing** — signals fire at decision points (breakouts, boundary touches), not during formation

### 3.1 Support / Resistance

**File:** `src/patterns/support_resistance.py`

**Old behavior:** Flagged every bar where Close was within 0.3×ATR of the rolling 50-bar max (resistance) or min (support). No filtering.

**Problems:**
- In an uptrend, the rolling max rises continuously → resistance is always near the current price → near_resistance fires almost every day
- No cooldown → same level fires 5–10 days in a row

**Changes made:**

| Parameter | Before | After | Rationale |
|-----------|--------|-------|-----------|
| `stability_window` | — | **3** | Resistance/support must be flat (unchanged) for at least 3 bars. A continuously rising "resistance" in an uptrend is not a real level — it's just the trend. True resistance means the *same price zone* is tested repeatedly. |
| `cooldown` | — | **10** | After a signal fires, suppress for 10 trading days. One signal per approach to a level. |

**Result:** 672 → 40 events (−94%), consecutive-day clustering eliminated.

In [ ]:
# Run the tuned S/R detector
df_sr = calculate_support_resistance(df, window=50, atr_mult=0.3,
                                     cooldown=10, stability_window=3)

nr_new = df_sr['near_resistance']
ns_new = df_sr['near_support']

print(f'near_resistance: {old_nr.sum()} → {nr_new.sum()}  '
      f'({100*(1 - nr_new.sum()/old_nr.sum()):.0f}% reduction)')
print(f'near_support:    {old_ns.sum()} → {ns_new.sum()}')

# Clustering check
new_nr_diffs = df_sr.index[nr_new].to_series().diff().dt.days
print(f'\nConsecutive-day clustering: {int((new_nr_diffs == 1).sum())} '
      f'(was {int((old_nr_diffs == 1).sum())})')

In [ ]:
# Plot: S/R detections overlaid on price (sample period for visibility)
start, end = '2018-01-01', '2020-06-30'
mask = (df_sr.index >= start) & (df_sr.index <= end)
sub = df_sr.loc[mask]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sub.index, sub['Close'], color='#1f77b4', linewidth=0.8, label='SPY Close')
ax.plot(sub.index, sub['resistance'], color='#d62728', linewidth=0.6,
        linestyle='--', alpha=0.5, label='Resistance level')
ax.plot(sub.index, sub['support'], color='#2ca02c', linewidth=0.6,
        linestyle='--', alpha=0.5, label='Support level')

# Mark detections
nr_dates = sub.index[sub['near_resistance']]
ns_dates = sub.index[sub['near_support']]
if len(nr_dates) > 0:
    ax.scatter(nr_dates, sub.loc[nr_dates, 'Close'], marker='v', color='#d62728',
               s=80, zorder=5, label=f'Near resistance ({len(nr_dates)})')
if len(ns_dates) > 0:
    ax.scatter(ns_dates, sub.loc[ns_dates, 'Close'], marker='^', color='#2ca02c',
               s=80, zorder=5, label=f'Near support ({len(ns_dates)})')

ax.set_title(f'Support/Resistance Detections — {start} to {end} (tuned)', fontsize=13)
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:** The stability filter removes the continuous stream of signals during uptrends. Signals now appear only when price approaches a level that has been established and unchanged for several bars — which is closer to how a human trader identifies support and resistance.

### 3.2 Channels

**File:** `src/patterns/channels.py`

**Old behavior:** Fit linear regressions to highs and lows, checked parallelism and width, counted any bar near the trendline as a "touch." No fit quality check.

**Problems:**
- Touch counting was inflated: 10 consecutive bars near the upper line counted as 10 touches
- No R² check: regression can fit a "channel" to any trending data, even if price doesn't actually bounce between the lines
- No cooldown: once near a boundary, fired every day

**Changes made:**

| Parameter | Before | After | Rationale |
|-----------|--------|-------|-----------|
| `r2_min` | — | **0.70** | Both trendline regressions must explain ≥70% of variance. This ensures price actually tracks the channel structure, not just any trending data. |
| `min_touches` | 3 | **4** | Require more distinct reversals at each boundary. |
| Touch counting | consecutive bars | **distinct reversals** (≥5 bars apart) | A cluster of bars near the upper line is ONE touch, not ten. |
| `cooldown` | — | **10** | One signal per approach to a channel boundary. |

**Supervisor reference:** The OTP trend channel (Oct 2025 – Jan 2026) demonstrates what a well-formed channel looks like: ~80 bars of structured price action with clear bounces between parallel trendlines. The R² and distinct-touch filters enforce this quality standard.

**Result:** 235 → 43 events (−82%), all `channel_down` detections removed (poor R²), consecutive-day clustering eliminated.

In [ ]:
from src.patterns.channels import detect_channel

df_ch = detect_channel(df, window=50, min_touches=4, r2_min=0.70, cooldown=10)
ch = df_ch['channel_pattern']

ch_up = (ch == 'channel_up').sum()
ch_down = (ch == 'channel_down').sum()
print(f'channel_up:   {ch_up}')
print(f'channel_down: {ch_down}')
print(f'Total:        {ch.notna().sum()}')

# Clustering
ch_dates = df_ch.index[ch.notna()]
ch_diffs = ch_dates.to_series().diff().dt.days
print(f'Consecutive-day clustering: {int((ch_diffs == 1).sum())}')

In [ ]:
# Plot: channel detections on price
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['Close'], color='#1f77b4', linewidth=0.7, label='SPY Close')

ch_event_dates = df_ch.index[ch.notna()]
if len(ch_event_dates) > 0:
    ax.scatter(ch_event_dates, df.loc[ch_event_dates, 'Close'],
               marker='D', color='#ff7f0e', s=40, zorder=5,
               label=f'Channel events ({len(ch_event_dates)})')

ax.set_title('Channel Detections Over Full History (tuned: R²≥0.70, distinct touches)', fontsize=13)
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:** Channel detections are now sparse and spread across the full 16-year history. The R² filter ensures that only well-structured channels pass — periods where price genuinely oscillated between parallel trendlines rather than just trending in one direction.

### 3.3 Triangles

**File:** `src/patterns/triangles.py`

**Old behavior:** Fit regression to highs/lows, required ≥3% compression, fired on breakout bar. No cooldown.

**Problems:**
- Weak compression threshold (3%) triggered on minor consolidations
- Same breakout fired on 2–4 consecutive days as price moved beyond the threshold
- No signal for the "upper-limit test" setup (supervisor reference)

**Changes made:**

| Parameter | Before | After | Rationale |
|-----------|--------|-------|-----------|
| `min_convergence_pct` | 0.03 (3%) | **0.05 (5%)** | Require more meaningful range compression to reduce weak formations. |
| `cooldown` | — | **10** | One signal per breakout. |
| **New signal type** | — | `desc_triangle_upper_test` | Detects price approaching the falling upper trendline within a descending triangle — the setup a discretionary trader watches before a potential breakdown. |

**Supervisor reference:** The OTP descending triangle upper-limit test illustrates the pattern: price approaches the descending upper resistance line for another test. This is the decision point a human trader would act on.

**Result:** 80 → 38 events (−52%), new signal type added, consecutive-day clustering eliminated.

In [ ]:
from src.patterns.triangles import detect_triangle_pattern

df_tri = detect_triangle_pattern(df, window=50, min_convergence_pct=0.05, cooldown=10)
tri = df_tri['triangle_pattern']

print('Triangle events by type:')
for t in sorted(tri.dropna().unique()):
    print(f'  {t}: {(tri == t).sum()}')
print(f'  Total: {tri.notna().sum()}')

tri_diffs = df_tri.index[tri.notna()].to_series().diff().dt.days
print(f'\nConsecutive-day clustering: {int((tri_diffs == 1).sum())}')

In [ ]:
# Plot: triangle detections on price
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['Close'], color='#1f77b4', linewidth=0.7, label='SPY Close')

colors = {
    'ascending_triangle': '#2ca02c',
    'descending_triangle': '#d62728',
    'symmetric_triangle': '#9467bd',
    'desc_triangle_upper_test': '#ff7f0e',
}
markers = {
    'ascending_triangle': '^',
    'descending_triangle': 'v',
    'symmetric_triangle': 's',
    'desc_triangle_upper_test': 'o',
}

for ttype in sorted(tri.dropna().unique()):
    dates = df_tri.index[tri == ttype]
    ax.scatter(dates, df.loc[dates, 'Close'],
               marker=markers.get(ttype, 'o'), color=colors.get(ttype, 'gray'),
               s=60, zorder=5, label=f'{ttype} ({len(dates)})')

ax.set_title('Triangle Detections Over Full History (tuned: 5% compression, cooldown)', fontsize=13)
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:** Triangle detections are now rare and well-separated. The new `desc_triangle_upper_test` signal (orange circles) captures the supervisor's reference pattern — price testing the descending upper trendline — without requiring a full breakout.

### 3.4 Multiple Tops / Bottoms

**File:** `src/patterns/multiple_tops_bottoms.py`

**Old behavior:** Used rolling max/min of highs/lows to detect ceiling/floor conditions, confirmed by a 3-bar close slope. No cooldown.

**Problems:**
- 3-bar close slope was too short — a single noisy bar could flip the trend direction
- No cooldown → mild clustering (17% within 5 days)

**Changes made:**

| Parameter | Before | After | Rationale |
|-----------|--------|-------|-----------|
| `confirm_bars` | 3 | **5** | Longer trend confirmation window reduces noise from short-term fluctuations. |
| `cooldown` | — | **10** | One signal per top/bottom formation. |

**Result:** 87 → 63 events (−28%), consecutive-day clustering eliminated.

In [ ]:
from src.patterns.multiple_tops_bottoms import detect_multiple_tops_bottoms

df_mtb = detect_multiple_tops_bottoms(df, window=50, confirm_bars=5, cooldown=10)
mtb = df_mtb['multiple_top_bottom_pattern']

mt = (mtb == 'multiple_top').sum()
mb = (mtb == 'multiple_bottom').sum()
print(f'multiple_top:    {mt}')
print(f'multiple_bottom: {mb}')
print(f'Total:           {mtb.notna().sum()}')

mtb_diffs = df_mtb.index[mtb.notna()].to_series().diff().dt.days
print(f'\nConsecutive-day clustering: {int((mtb_diffs == 1).sum())}')

In [ ]:
# Plot: multi top/bottom detections on price
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['Close'], color='#1f77b4', linewidth=0.7, label='SPY Close')

mt_dates = df_mtb.index[mtb == 'multiple_top']
mb_dates = df_mtb.index[mtb == 'multiple_bottom']

if len(mt_dates) > 0:
    ax.scatter(mt_dates, df.loc[mt_dates, 'Close'], marker='v', color='#d62728',
               s=60, zorder=5, label=f'Multiple top ({len(mt_dates)})')
if len(mb_dates) > 0:
    ax.scatter(mb_dates, df.loc[mb_dates, 'Close'], marker='^', color='#2ca02c',
               s=60, zorder=5, label=f'Multiple bottom ({len(mb_dates)})')

ax.set_title('Multiple Tops/Bottoms Over Full History (tuned: 5-bar confirmation, cooldown)', fontsize=13)
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation:** Multiple-top signals cluster around market peaks (2020 pre-COVID, 2022 bear market onset), while multiple-bottom signals appear during corrections and recoveries. The pattern is directionally consistent with fundamental market dynamics.

---

## 4. Summary Table — All Tuning Changes

The following table consolidates every change made during the tuning phase.

In [ ]:
tuning_summary = pd.DataFrame([
    {
        'Detector': 'Support/Resistance',
        'File': 'support_resistance.py',
        'Key changes': 'stability_window=3, cooldown=10',
        'Before': 672,
        'After': 40,
        'Reduction': '−94%',
        'Clustering': '0 consecutive',
    },
    {
        'Detector': 'Channels',
        'File': 'channels.py',
        'Key changes': 'R²≥0.70, distinct touches (5-bar sep.), min_touches=4, cooldown=10',
        'Before': 235,
        'After': 43,
        'Reduction': '−82%',
        'Clustering': '0 consecutive',
    },
    {
        'Detector': 'Triangles',
        'File': 'triangles.py',
        'Key changes': 'compression 3%→5%, cooldown=10, new desc_triangle_upper_test',
        'Before': 80,
        'After': 38,
        'Reduction': '−52%',
        'Clustering': '0 consecutive',
    },
    {
        'Detector': 'Multi Top/Bottom',
        'File': 'multiple_tops_bottoms.py',
        'Key changes': 'confirm_bars 3→5, cooldown=10',
        'Before': 87,
        'After': 63,
        'Reduction': '−28%',
        'Clustering': '0 consecutive',
    },
    {
        'Detector': 'TOTAL',
        'File': 'scanner.py',
        'Key changes': 'Per-detector window params; unified event flag',
        'Before': 1067,
        'After': 183,
        'Reduction': '−83%',
        'Clustering': '0 consecutive',
    },
])

display(tuning_summary.style
        .hide(axis='index')
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))

### Additional file changes

| File | Change |
|------|--------|
| `scanner.py` | Changed from single `window` argument to per-detector window parameters |
| `evaluate_rates.py` | Updated to use `window=50` for all detectors |
| `validation.py` | New module — `EventValidator` class for visual and statistical pattern quality assessment |
| `utils/plotting.py` | New module — shared candlestick chart and annotation helpers |
| `03_pattern_validation.ipynb` | Updated with tuned scanner call, added `desc_triangle_upper_test` visual checks |

---

## 5. Overall Event-Rate Outcome

We now run the complete unified scanner to confirm the final event counts and overall rate.

In [ ]:
df_final = scan_all_patterns(df)
validator = EventValidator(df_final, context_bars=30, forward_horizon=10)

total_bars = len(df_final)
total_events = len(validator.events)
event_rate = total_events / total_bars

print(f'Total trading days: {total_bars:,}')
print(f'Total events:       {total_events}')
print(f'Event rate:         {event_rate:.1%}')
print(f'Events per year:    {total_events / 16:.1f}  (over ~16 years of data)')

In [ ]:
# Breakdown by event type
breakdown = validator.event_density_by_detector()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = plt.cm.Set2(np.linspace(0, 1, len(breakdown)))
breakdown.plot.pie(ax=axes[0], autopct='%1.0f%%', startangle=90,
                   colors=colors, textprops={'fontsize': 9})
axes[0].set_ylabel('')
axes[0].set_title('Event Type Distribution')

# Bar chart
breakdown.sort_values().plot.barh(ax=axes[1], color='steelblue')
axes[1].set_xlabel('Number of Events')
axes[1].set_title('Event Counts by Detector')

plt.tight_layout()
plt.show()

In [ ]:
# Before vs After comparison bar chart
detectors = ['S/R', 'Channels', 'Triangles', 'Multi T/B']
before = [672, 235, 80, 87]
after = [40, 43, 38, 63]

x = np.arange(len(detectors))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, before, width, label='Before tuning', color='#d62728', alpha=0.7)
bars2 = ax.bar(x + width/2, after, width, label='After tuning', color='#2ca02c', alpha=0.7)

ax.set_ylabel('Number of Events')
ax.set_title('Event Counts Before vs. After Tuning')
ax.set_xticks(x)
ax.set_xticklabels(detectors)
ax.legend()

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly event density over time
density = validator.event_density_by_month()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(density.index, density['event_count'], width=25, color='steelblue', alpha=0.7)
ax.set_ylabel('Events per Month')
ax.set_title('Monthly Event Count (tuned detectors)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Average events per month: {density["event_count"].mean():.1f}')
print(f'Months with 0 events:     {(density["event_count"] == 0).sum()} / {len(density)}')

### Section conclusion

The tuned detection layer produces **183 events** over 16 years (4.5% event rate, ~11 events/year). This is deliberately sparse:

- **Zero consecutive-day clustering** — each event is a distinct pattern, not a repetition
- Events are distributed across the full history, not concentrated in particular periods
- The event rate is low enough that the ML model will only evaluate high-conviction setups, but high enough to provide meaningful training data across different market regimes

---

## 6. Visual Validation

The `EventValidator` class (in `src/patterns/validation.py`) provides tools for visual quality assessment. Below we show representative events from each detector type, plotted with candlestick context, support/resistance levels, and forward-return annotations.

This section directly addresses the supervisor's requirement that detections should **match what a human trader would reasonably identify** on a chart.

In [ ]:
# near_resistance examples
fig = validator.plot_sample(detector='near_resistance', n=4, cols=2, seed=42)
if fig:
    fig.suptitle('Near Resistance — Sample Events', fontsize=14, y=1.02)
    plt.show()

In [ ]:
# channel_up examples
fig = validator.plot_sample(detector='channel_up', n=4, cols=2, seed=42)
if fig:
    fig.suptitle('Channel (Up) — Sample Events', fontsize=14, y=1.02)
    plt.show()

In [ ]:
# descending_triangle examples
fig = validator.plot_sample(detector='descending_triangle', n=4, cols=2, seed=42)
if fig:
    fig.suptitle('Descending Triangle Breakout — Sample Events', fontsize=14, y=1.02)
    plt.show()

In [ ]:
# desc_triangle_upper_test examples (supervisor reference pattern)
fig = validator.plot_sample(detector='desc_triangle_upper_test', n=4, cols=2, seed=42)
if fig:
    fig.suptitle('Descending Triangle Upper-Limit Test — Sample Events\n'
                 '(supervisor reference pattern)', fontsize=14, y=1.02)
    plt.show()
else:
    print(f'desc_triangle_upper_test: {(validator.events["event_type"] == "desc_triangle_upper_test").sum()} events (all shown in triangle section)')

In [ ]:
# multiple_bottom examples
fig = validator.plot_sample(detector='multiple_bottom', n=4, cols=2, seed=42)
if fig:
    fig.suptitle('Multiple Bottom — Sample Events', fontsize=14, y=1.02)
    plt.show()

In [ ]:
# multiple_top examples
fig = validator.plot_sample(detector='multiple_top', n=4, cols=2, seed=42)
if fig:
    fig.suptitle('Multiple Top — Sample Events', fontsize=14, y=1.02)
    plt.show()

### Forward-return quality by detector

In [ ]:
report = validator.quality_report()
display(report.round(3).style
        .hide(axis='index')
        .set_properties(**{'text-align': 'right'})
        .set_properties(subset=['detector'], **{'text-align': 'left'}))

In [ ]:
# Box plot of forward returns by detector
event_types = [t for t in validator.get_detector_types()]
data_to_plot = []
labels = []
for etype in event_types:
    fwd = validator.events.loc[
        validator.events['event_type'] == etype, 'fwd_return'
    ].dropna() * 100
    if len(fwd) >= 3:
        data_to_plot.append(fwd.values)
        labels.append(f'{etype}\n(n={len(fwd)})')

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(data_to_plot, labels=labels, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5),
                medianprops=dict(color='#d62728', linewidth=2))
ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_ylabel('10-bar Forward Return (%)')
ax.set_title('Forward-Return Distributions by Detector Type')
plt.xticks(rotation=30, ha='right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Section conclusion

The visual spot-checks confirm that tuned detections correspond to structurally meaningful chart patterns:

- **Resistance signals** appear at genuine horizontal levels that have been tested and held, not at continuously rising trend highs.
- **Channel events** correspond to price action within well-defined parallel trendlines with multiple distinct boundary touches.
- **Triangle breakouts** require meaningful range compression (≥5%) before the breakout is flagged.
- **Multiple tops/bottoms** fire at price zones that have been tested repeatedly, with 5-bar trend confirmation.

Forward returns are directionally positive across most detector types, with `multiple_top` and `descending_triangle` showing the strongest signals.

---

## 7. Why This Phase Matters for the Full Thesis

The pattern detection layer is the **foundation** of the entire project pipeline. Every subsequent phase depends on its quality:

```
Pattern Detection  →  HMM Regimes  →  Event Dataset  →  Features  →  ML Models  →  Backtest
     (Phase 3)         (Phase 4)       (Phase 5)        (Phase 6)    (Phase 7)     (Phase 8)
```

If the event detection layer is noisy, the contamination propagates downstream:

| If events are noisy... | Then... |
|------------------------|----------|
| Clustered signals | ML model trains on redundant, correlated samples → inflated performance |
| False-positive patterns | Labels become unreliable → model learns noise |
| Dense signals (26% rate) | System trades too often → transaction costs erode returns |
| No structural validation | Features reflect statistical artifacts, not real patterns |

By tuning the detectors to produce **183 sparse, non-clustered, structurally validated events**, we ensure that:

1. **Feature engineering** (Phase 6) operates on meaningful pattern context
2. **Trade labels** (Phase 5) reflect genuine entry points, not noise
3. **ML models** (Phase 7) learn from clean signal, not artifact
4. **Walk-forward backtesting** (Phase 8) produces realistic trade frequency and turnover

### Readiness for Phase 4: HMM Regime Detection

With the event detection layer validated, the next phase is fitting a **Gaussian Hidden Markov Model** to daily log returns and rolling volatility. The HMM will identify 2–3 market regimes (e.g., bull/bear/sideways or low-vol/high-vol), which will be joined back to the event dataset as contextual features.

The regime labels serve as a **filter and context** for the ML models — the hypothesis is that the same technical pattern (e.g., a support touch) has different expected outcomes depending on whether the market is in a bullish or bearish regime.

---

## 8. All Detected Events on Price — Full History

A single chart showing all 183 tuned events overlaid on the SPY price history.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df.index, df['Close'], color='#1f77b4', linewidth=0.6, alpha=0.8, label='SPY Close')

# Color and marker mapping for all event types
style_map = {
    'near_support':               ('^', '#2ca02c', 'Near support'),
    'near_resistance':            ('v', '#d62728', 'Near resistance'),
    'channel_up':                 ('D', '#ff7f0e', 'Channel up'),
    'ascending_triangle':         ('^', '#17becf', 'Asc. triangle'),
    'descending_triangle':        ('v', '#9467bd', 'Desc. triangle'),
    'desc_triangle_upper_test':   ('o', '#e377c2', 'Desc. tri. test'),
    'multiple_top':               ('x', '#8c564b', 'Multiple top'),
    'multiple_bottom':            ('+', '#bcbd22', 'Multiple bottom'),
}

for etype, (marker, color, label) in style_map.items():
    dates = validator.events.index[validator.events['event_type'] == etype]
    if len(dates) > 0:
        ax.scatter(dates, df.loc[dates, 'Close'], marker=marker, color=color,
                   s=45, zorder=5, label=f'{label} ({len(dates)})', alpha=0.85)

ax.set_title(f'All Tuned Pattern Detections — {total_events} events across {total_bars:,} bars ({event_rate:.1%})',
             fontsize=14)
ax.set_ylabel('Price ($)')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=8, ncol=2, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 9. Conclusion

### What was achieved

The pattern detection phase of the project has been **completed and validated**. Starting from an initial event rate of 26.5% (1,067 events, heavily clustered), the four detectors were systematically tuned to produce **183 high-quality events (4.5% rate) with zero consecutive-day clustering**.

The key improvements were:

1. **Level stability filter** for support/resistance — eliminates false signals in trending markets
2. **R² fit quality** and **distinct touch counting** for channels — ensures structural validity
3. **Stricter compression** and a **new upper-limit test signal** for triangles — captures the supervisor's reference pattern
4. **Longer trend confirmation** for multiple tops/bottoms — reduces short-term noise
5. **Universal cooldown** across all detectors — one signal per event, no repetitions

### Current project status

| Phase | Status |
|-------|--------|
| 1. Data pipeline | Complete |
| 2. Pattern detection | Complete |
| 3. Pattern validation & tuning | **Complete** |
| 4. HMM regime detection | Next |
| 5. Event dataset construction | Planned |
| 6. Feature engineering | Planned |
| 7. ML model training | Planned |
| 8. Walk-forward backtesting | Planned |
| 9. Final comparison & reporting | Planned |

### Next step

**Phase 4: HMM Regime Detection** — Fit a Gaussian Hidden Markov Model using daily log returns and rolling volatility to identify 2–3 market regimes. Regime labels will be joined to the event dataset as context for the ML models.